In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Mitigação de erros por redes tensoriais (TEM): Uma Qiskit Function da Algorithmiq
> **Note:** As Qiskit Functions são um recurso experimental disponível somente para usuários dos planos IBM Quantum&reg; Premium, Flex e On-Prem (via IBM Quantum Platform API). Elas estão em versão de pré-visualização e sujeitas a alterações.


<details>
<summary><b>Versões dos pacotes</b></summary>

O código desta página foi desenvolvido usando os seguintes requisitos.
Recomendamos usar estas versões ou mais recentes.

```
qiskit[all]~=2.3.0
qiskit-ibm-catalog~=0.11.0
```
</details>
## Visão geral
O método de Mitigação de Erros por Redes Tensoriais (TEM) da Algorithmiq é um algoritmo
híbrido quântico-clássico projetado para realizar a mitigação de ruído inteiramente na
etapa de pós-processamento clássico. Com o TEM, você pode calcular os
valores esperados de observáveis mitigando os inevitáveis erros induzidos por ruído
que ocorrem no hardware quântico, com maior precisão e eficiência de custo,
tornando-o uma opção altamente atrativa tanto para pesquisadores quânticos quanto para
profissionais da indústria.

O método consiste em construir uma rede tensorial que representa o inverso do
canal de ruído global que afeta o estado do processador quântico, e então
aplicar o mapa aos resultados de medição informacionalmente completos obtidos do
estado ruidoso para obter estimadores não viesados para os observáveis.

Como vantagem, o TEM aproveita medições informacionalmente completas para dar
acesso a um vasto conjunto de valores esperados mitigados de observáveis, e possui
sobrecarga de amostragem ótima no hardware quântico, conforme descrito em Filippov et
al. (2023), [arXiv:2307.11740](https://arxiv.org/abs/2307.11740), e Filippov
et al. (2024), [arXiv:2403.13542](https://arxiv.org/abs/2403.13542). A
sobrecarga de medição refere-se ao número de medições adicionais necessárias para
realizar a mitigação de erros eficiente, um fator crítico na viabilidade de
computações quânticas. Portanto, o TEM tem o potencial de possibilitar a
vantagem quântica em cenários complexos, como aplicações nos campos de caos quântico,
física de muitos corpos, dinâmica de Hubbard e simulações de química de moléculas pequenas.

As principais funcionalidades e benefícios do TEM podem ser resumidos como:

1.  **Sobrecarga de medição ótima**: O TEM é ótimo em relação aos
[limites teóricos](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.131.210601),
o que significa que nenhum método pode alcançar uma sobrecarga de medição menor. Em outras
palavras, o TEM requer o número mínimo de medições adicionais para realizar
a mitigação de erros. Isso, por sua vez, significa que o TEM usa o tempo de execução quântico mínimo.
2.  **Custo-benefício**: Como o TEM lida com a mitigação de ruído inteiramente na
etapa de pós-processamento, não há necessidade de adicionar circuitos extras ao computador
quântico, o que não só torna a computação mais barata, mas também reduz o
risco de introduzir erros adicionais devido às imperfeições dos dispositivos quânticos.
3.  **Estimativa de múltiplos observáveis**: Graças às medições informacionalmente completas,
o TEM estima eficientemente múltiplos observáveis com os mesmos
dados de medição do computador quântico.
4.  **Mitigação de erros de medição**: A Qiskit Function TEM também inclui um
[método proprietário de mitigação de erros de medição](https://journals.aps.org/prresearch/abstract/10.1103/PhysRevResearch.5.033154)
capaz de reduzir significativamente os erros de leitura após uma curta execução de calibração.
5.  **Precisão**: O TEM melhora significativamente a precisão e a confiabilidade de
simulações quânticas digitais, tornando os algoritmos quânticos mais exatos e
confiáveis.
## Descrição
A função TEM permite que você obtenha valores esperados com mitigação de erros para
múltiplos observáveis em um Circuit quântico com sobrecarga de amostragem mínima. O
Circuit é medido com uma medida de operador com valor positivo informacionalmente completa
(IC-POVM), e os resultados de medição coletados são processados em um
computador clássico. Essa medição é usada para realizar os métodos de rede tensorial
e construir um mapa de inversão de ruído. A função aplica um mapa que
inverte completamente todo o Circuit ruidoso usando redes tensoriais para representar as
camadas ruidosas.

![Esquema do TEM](../docs/images/guides/algorithmiq-tem/tem_scheme.svg "Estimativa com mitigação de erros de um observável O via pós-processamento dos resultados de medição do processador quântico ruidoso. U e N denotam uma operação quântica ideal e o mapa de ruído associado, que pode ser geralmente não-local (e estendido às caixas cinzas). D representa um tensor de operadores que são duais aos efeitos na medição IC. O módulo de mitigação de ruído M é uma rede tensorial que é contraída eficientemente de dentro para fora. A primeira iteração da contração é representada pela linha pontilhada roxa, a segunda pela linha tracejada, e a terceira pela linha sólida.")

Uma vez que os circuitos são enviados à função, eles são transpilados e
otimizados para minimizar o número de camadas com gates de dois Qubits (os gates
mais ruidosos em dispositivos quânticos). O ruído que afeta as camadas é aprendido por meio do
[Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/noise-learner-noise-learner)
usando um modelo de ruído esparso de Pauli-Lindblad, conforme descrito em E. van den Berg, Z.
Minev, A. Kandala, K. Temme, Nat. Phys. (2023).
[arXiv:2201.09866](https://arxiv.org/abs/2201.09866).

O modelo de ruído é uma descrição precisa do ruído no dispositivo, capaz de
capturar características sutis, incluindo a diafonia entre Qubits. No entanto, o ruído nos
dispositivos pode flutuar e derivar, e o ruído aprendido pode não ser preciso no
momento em que a estimativa é feita. Isso pode resultar em resultados imprecisos.
## Primeiros passos
Autentique-se usando sua [chave de API do IBM Quantum Platform](http://quantum.cloud.ibm.com/), e selecione a função TEM como a seguir. (Este trecho assume que você já [salvou sua conta](/guides/functions#install-qiskit-functions-catalog-client) em seu ambiente local.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

tem_function_name = "algorithmiq/tem"

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Load your function
tem = catalog.load(tem_function_name)

## Exemplo
O trecho a seguir mostra um exemplo em que o TEM é usado para calcular os valores esperados de um observável dado um Circuit quântico simples.

In [2]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

# Create a quantum circuit
qc = QuantumCircuit(3)
qc.u(0.4, 0.9, -0.3, 0)
qc.u(-0.4, 0.2, 1.3, 1)
qc.u(-1.2, -1.2, 0.3, 2)
for _ in range(2):
    qc.barrier()
    qc.cx(0, 1)
    qc.cx(2, 1)
    qc.barrier()
    qc.u(0.4, 0.9, -0.3, 0)
    qc.u(-0.4, 0.2, 1.3, 1)
    qc.u(-1.2, -1.2, 0.3, 2)

# Define the observables
observable = SparsePauliOp("IYX", 1.0)

# Define the execution options
pub = (qc, [observable])
options = {"default_precision": 0.02}

# Define backend to use. TEM will choose the least-busy device reported by IBM if not specified
backend_name = "ibm_torino"

job = tem.run(pubs=[pub], backend_name=backend_name, options=options)

Use as APIs do Qiskit Serverless para verificar o status da sua carga de trabalho do Qiskit Function:

In [3]:
print(job.status())

QUEUED


You can return the results as:

In [4]:
result = job.result()
evs = result[0].data.evs

Você pode retornar os resultados da seguinte forma:

In [5]:
print(job.result())

PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(1,), dtype=float64>), stds=np.ndarray(<shape=(1,), dtype=float64>)), metadata={'evs_non_mitigated': array([-0.06314623]), 'stds_non_mitigated': array([0.05917147]), 'evs_mitigated_no_readout_mitigation': array([-0.06411205]), 'stds_mitigated_no_readout_mitigation': array([0.05992467]), 'evs_non_mitigated_with_readout_mitigation': array([-0.07028881]), 'stds_non_mitigated_with_readout_mitigation': array([0.06353934])})], metadata={'resource_usage': {'RUNNING: OPTIMIZING_FOR_HARDWARE': {'CPU_TIME': 0.915754}, 'RUNNING: WAITING_FOR_QPU': {'CPU_TIME': 18.804865}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 10.433445}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 159.0}}})


> **Info:** O valor esperado para o Circuit sem ruído para o operador dado deve ser em torno de `0.18409094298943401`.
## Entradas
**Parâmetros**

Nome | Tipo | Descrição | Obrigatório | Padrão | Exemplo
-- | -- | -- | -- | -- | --
`pubs` | Iterable[EstimatorPubLike] | Um iterável de objetos do tipo PUB (bloco unificado de primitiva), como tuplas `(circuit, observables)` ou `(circuit, observables, parameters, precision)`. Veja [Visão geral dos PUBs](/guides/primitive-input-output#overview-of-pubs) para mais informações. Se um Circuit não-ISA for passado, ele será transpilado com configurações ótimas. Se um Circuit ISA for passado, ele não será transpilado; nesse caso, o observável deve ser definido em todo o QPU. | Sim | N/A | (circuit, observables)
`backend_name` | str | Nome do Backend para realizar a consulta. | Não | Se não for fornecido, o Backend menos ocupado será usado. | "ibm_fez"
`options` | dict | Opções de entrada. Veja a seção `Options` para mais detalhes. | Não | Veja a seção `Options` para mais detalhes. | {"max_bond_dimension": 100\}

> **Caution:** O TEM atualmente possui as seguintes limitações:
> 
>   - Circuitos parametrizados não são suportados. O argumento de parâmetros deve ser definido como `None` se a precisão for especificada. Essa restrição será removida em versões futuras.
>   - Apenas circuitos sem loops são suportados. Essa restrição será removida em versões futuras.
>   - Gates não-unitários, como reset, measure e todas as formas de fluxo de controle, não são suportados. O suporte para reset será adicionado em versões futuras.
### Opções
Um dicionário contendo as opções avançadas para o TEM. O dicionário pode conter as chaves da tabela a seguir. Se alguma das opções não for fornecida, o valor padrão listado na tabela será usado. Os valores padrão são adequados para o uso típico do TEM.

Nome | Opções | Descrição | Padrão
-- | -- | -- | --
`tem_max_bond_dimension` | int | A dimensão de ligação máxima a ser usada para as redes tensoriais. | 500 |
`tem_compression_cutoff` | float | O valor de corte a ser usado para as redes tensoriais. | 1e-16
`compute_shadows_bias_from_observable` | bool | Um sinalizador booleano indicando se o viés para o protocolo de medição de sombras clássicas deve ser adaptado ao observável do PUB ou não. Se False, o protocolo de sombras clássicas (probabilidade igual de medir Z, X, Y) será usado. | False
`shadows_bias` | np.ndarray | O viés a ser usado para o protocolo de medição de sombras clássicas aleatórias, um array 1d ou 2d de tamanho 3 ou forma (num_qubits, 3) respectivamente. A ordem é ZXY | np.array([1 / 3, 1 / 3, 1 / 3])
`max_execution_time` | int ou `None` | O tempo máximo de execução no QPU em segundos. Se o tempo de execução exceder esse valor, o job será cancelado. Se `None`, será aplicado um limite padrão definido pelo Qiskit Runtime. | `None`
`num_randomizations` | int | O número de randomizações a ser usado para aprendizado de ruído e entrelaçamento de gates. | 32
`max_layers_to_learn` | int | O número máximo de camadas únicas a aprender. | 4
`mitigate_readout_error` | bool | Um sinalizador booleano indicando se a mitigação de erros de leitura deve ser realizada ou não. | True
`num_readout_calibration_shots` | int | O número de shots a ser usado para a mitigação de erros de leitura. | 10000
`default_precision` | float | A precisão padrão a ser usada para os PUBs para os quais a precisão não é especificada. | 0.02
`seed` | int ou `None` | Define a semente do gerador de números aleatórios para reprodutibilidade. Se `None`, não define a semente. | None
## Saídas
Um [PrimitiveResults](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) do Qiskit contendo o resultado mitigado pelo TEM. O resultado para cada PUB é retornado como um [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) contendo os seguintes campos:

Nome | Tipo | Descrição
-- | -- | --
data | DataBin | Um [DataBin](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin) do Qiskit contendo o observável mitigado pelo TEM e seu erro padrão. O DataBin possui os seguintes campos: <ul><li>`evs`: O valor do observável mitigado pelo TEM.</li><li>`stds`: O erro padrão do observável mitigado pelo TEM.</li></ul>
metadata | dict | Um dicionário contendo resultados adicionais. O dicionário contém as seguintes chaves: <ul><li>`"evs_non_mitigated"`: O valor do observável sem mitigação de erros.</li><li>`"stds_non_mitigated"`: O erro padrão do resultado sem mitigação de erros.</li><li>`"evs_mitigated_no_readout_mitigation"`: O valor do observável com mitigação de erros, mas sem mitigação de erros de leitura.</li><li>`"stds_mitigated_no_readout_mitigation"`: O erro padrão do resultado com mitigação de erros, mas sem mitigação de erros de leitura.</li><li>`"evs_non_mitigated_with_readout_mitigation"`: O valor do observável sem mitigação de erros, mas com mitigação de erros de leitura.</li><li>`"stds_non_mitigated_with_readout_mitigation"`: O erro padrão do resultado sem mitigação de erros, mas com mitigação de erros de leitura.</li></ul>
## Obtendo mensagens de erro
Se o status da sua carga de trabalho for ERROR, use job.result() para obter a mensagem de erro da seguinte forma: